In [0]:
hchb_cash_invoice = dbutils.widgets.get("hchb_cash_invoice")
cash_invoice=dbutils.widgets.get("cash_invoice")
episodic_line_items_revenue=dbutils.widgets.get("episodic_line_items_revenue")
refunds_all=dbutils.widgets.get("refunds_all")
payor_sources=dbutils.widgets.get("payor_sources")
deposits=dbutils.widgets.get("deposits")
payor_types=dbutils.widgets.get("payor_types")
usable_cash=dbutils.widgets.get("usable_cash")
cash_initialamounts_by_closing_period=dbutils.widgets.get("cash_initialamounts_by_closing_period")
closing_periods=dbutils.widgets.get("closing_periods")
yrinvo=dbutils.widgets.get("yrinvo")
client_episode_fs=dbutils.widgets.get("client_episode_fs")
client_episodes_all=dbutils.widgets.get("client_episodes_all")
agencies_servicelines_branches=dbutils.widgets.get("agencies_servicelines_branches")
agencies=dbutils.widgets.get("agencies")
companies=dbutils.widgets.get("companies")
officemapping =dbutils.widgets.get("officemapping")
datedimension=dbutils.widgets.get("datedimension")
office=dbutils.widgets.get("office")
date_path=dbutils.widgets.get("date_path")
cubeserviceofficetxnsourcesystem=dbutils.widgets.get("cubeserviceofficetxnsourcesystem")
payerdimension=dbutils.widgets.get("payerdimension")
client=dbutils.widgets.get("client")
stg_hchb_cash=dbutils.widgets.get("stg_hchb_cash")

In [0]:
spark.sql(
    f"""
    TRUNCATE TABLE {hchb_cash_invoice}
    """
)

In [0]:
spark.sql(
    f"""
CREATE OR REPLACE TEMP VIEW hchb_parameters AS
SELECT
  2 AS parmDateType,  
  CAST(NULL AS INT) AS parmDepositOption,
  date_sub(current_date(), 2190) AS parmDateFrom, 
  current_date() AS parmDateTo;
"""
)

spark.sql(
    f"""
CREATE OR REPLACE TEMP VIEW hchb_pps_invoices AS
SELECT DISTINCT 
  lir.elir_invoiceid AS invnum,
  CASE WHEN lir.elir_finalclaim = FALSE THEN 'RAP' ELSE 'Final' END AS invtype
FROM {episodic_line_items_revenue} lir
WHERE lir.elir_invoiceid IS NOT NULL;
"""
)

spark.sql(
    f"""
CREATE OR REPLACE TEMP VIEW hchb_refunds_data AS
SELECT
  rf.r_from_cashid AS rf_hsid,
  SUM(rf.r_balance) AS rf_amount,
  MAX(CASE 
    WHEN (rf.r_postdate >= (SELECT parmDateFrom FROM hchb_parameters) OR (SELECT parmDateFrom FROM hchb_parameters) IS NULL)
    AND (rf.r_postdate < date_add((SELECT parmDateTo FROM hchb_parameters), 1) OR (SELECT parmDateTo FROM hchb_parameters) IS NULL)
    THEN 1 ELSE 0 
  END) AS rf_hasrefundinpostdates,
  SUM(CASE 
    WHEN (rf.r_postdate >= (SELECT parmDateFrom FROM hchb_parameters) OR (SELECT parmDateFrom FROM hchb_parameters) IS NULL)
    AND (rf.r_postdate < date_add((SELECT parmDateTo FROM hchb_parameters), 1) OR (SELECT parmDateTo FROM hchb_parameters) IS NULL)
    THEN rf.r_balance ELSE 0 
  END) AS rf_amount_bypostdate
FROM {refunds_all} rf
GROUP BY rf.r_from_cashid;
"""
)

spark.sql(
    f"""
CREATE OR REPLACE TEMP VIEW hchb_results_payments AS
SELECT
  1 AS sortorder,
  c.company_name AS company,
  ag.agency_name AS agency,
  tp.d_branchcode AS branch,
  CONCAT(
    COALESCE(ce.epi_lastname, ''), 
    COALESCE(CONCAT(', ', ce.epi_firstname), ''), 
    COALESCE(CONCAT(' ', ce.epi_mi), '')
  ) AS client_name,
  COALESCE(ce.epi_mrnum, '') AS medical_record_number,
  pt.pt_desc AS payor_type_desc,
  ps.ps_desc AS payor_source,
  COALESCE(hp.uc_invoiceid, 0) AS invoice_number,
  CASE 
    WHEN ppsi.invtype = 'RAP' 
    THEN CAST(COALESCE(cibcp.cibcp_initialamount, 0) AS DECIMAL(15,2))
    ELSE 0 
  END AS rap_payment,
  CASE 
    WHEN ppsi.invtype = 'Final' 
    THEN CAST(COALESCE(cibcp.cibcp_initialamount, 0) AS DECIMAL(15,2))
    ELSE 0 
  END AS final_payment,
  CASE 
    WHEN ppsi.invtype IN ('RAP', 'Final') OR hp.uc_cashtypeid = 3 
    THEN 0
    ELSE COALESCE(cibcp.cibcp_initialamount, 0)
  END AS other_payment,
  COALESCE(rf.rf_amount, 0) AS refund_payment,
  CASE 
    WHEN hp.uc_cashtypeid = 3 
    THEN CAST(cibcp.cibcp_initialamount AS DECIMAL(15,2))
    ELSE 0 
  END AS unapplied_cash,
  CAST(tp.d_postdate AS DATE) AS deposit_date,
  CAST(tp.d_depositnumber AS STRING) AS deposit_number,
  tp.d_checknumber AS check_number,
  CAST(hp.uc_insertdate AS TIMESTAMP) AS date_entered,
  cper.cp_period AS month_end_close_reporting_period,
  ps.ps_id AS psid
FROM {deposits} tp
INNER JOIN {payor_sources} ps ON ps.ps_id = tp.d_payorsourceid
INNER JOIN {payor_types} pt ON pt.pt_id = ps.ps_ptid
INNER JOIN {usable_cash} hp ON tp.d_id = hp.uc_depositid
INNER JOIN {cash_initialamounts_by_closing_period} cibcp ON cibcp.cibcp_cashid = hp.uc_id
LEFT JOIN {closing_periods} cper ON cper.cp_id = cibcp.cibcp_closingperiodid
LEFT JOIN {yrinvo} y ON y.invnum = hp.uc_invoiceid
LEFT JOIN {client_episode_fs} cefs ON cefs.cefs_id = y.primid
LEFT JOIN {client_episodes_all} ce ON ce.epi_id = cefs.cefs_epiid
LEFT JOIN {agencies_servicelines_branches} asb ON asb.asb_slid = ce.epi_slid AND asb.asb_branchcode = tp.d_branchcode
LEFT JOIN {agencies} ag ON ag.agency_id = asb.asb_agencyid
LEFT JOIN {companies} c ON c.company_id = ag.agency_companyid
LEFT JOIN hchb_pps_invoices ppsi ON ppsi.invnum = hp.uc_invoiceid
LEFT JOIN hchb_refunds_data rf ON rf.rf_hsid = hp.uc_id
CROSS JOIN hchb_parameters p
WHERE (
  (
    (hp.uc_insertdate >= p.parmDateFrom OR p.parmDateFrom IS NULL)
    AND (hp.uc_insertdate < date_add(p.parmDateTo, 1) OR p.parmDateTo IS NULL)
    AND p.parmDateType = 2
  )
  OR
  (
    (hp.uc_postdate >= p.parmDateFrom OR p.parmDateFrom IS NULL)
    AND (hp.uc_postdate < date_add(p.parmDateTo, 1) OR p.parmDateTo IS NULL)
    AND p.parmDateType = 1
  )
);
"""
)

spark.sql(
    f"""
CREATE OR REPLACE TEMP VIEW hchb_results_refunds AS
SELECT
  1 AS sortorder,
  c.company_name AS company,
  ag.agency_name AS agency,
  tp.d_branchcode AS branch,
  CONCAT(
    COALESCE(ce.epi_lastname, ''), 
    COALESCE(CONCAT(', ', ce.epi_firstname), ''), 
    COALESCE(CONCAT(' ', ce.epi_mi), '')
  ) AS client_name,
  COALESCE(ce.epi_mrnum, '') AS medical_record_number,
  pt.pt_desc AS payor_type_desc,
  ps.ps_desc AS payor_source,
  COALESCE(hp.uc_invoiceid, 0) AS invoice_number,
  0 AS rap_payment,
  0 AS final_payment,
  0 AS other_payment,
  (-1 * COALESCE(r.rf_amount_bypostdate, 0)) AS refund_payment,
  CASE 
    WHEN hp.uc_cashtypeid = 3 
    THEN CAST(cibcp.cibcp_initialamount AS DECIMAL(15,2))
    ELSE 0 
  END AS unapplied_cash,
  CAST(tp.d_postdate AS DATE) AS deposit_date,
  CONCAT('R', CAST(tp.d_depositnumber AS STRING)) AS deposit_number,
  tp.d_checknumber AS check_number,
  CAST(hp.uc_insertdate AS TIMESTAMP) AS date_entered,
  cper.cp_period AS month_end_close_reporting_period,
  ps.ps_id AS psid
FROM {deposits} tp
INNER JOIN {payor_sources} ps ON ps.ps_id = tp.d_payorsourceid
INNER JOIN {payor_types} pt ON pt.pt_id = ps.ps_ptid
INNER JOIN {usable_cash} hp ON tp.d_id = hp.uc_depositid
INNER JOIN hchb_refunds_data r ON r.rf_hsid = hp.uc_id
LEFT JOIN {cash_initialamounts_by_closing_period} cibcp ON cibcp.cibcp_cashid = hp.uc_id
LEFT JOIN {closing_periods} cper ON cper.cp_id = cibcp.cibcp_closingperiodid
LEFT JOIN {yrinvo} y ON y.invnum = hp.uc_invoiceid
LEFT JOIN {client_episode_fs} cefs ON cefs.cefs_id = y.primid
LEFT JOIN {client_episodes_all} ce ON ce.epi_id = cefs.cefs_epiid
LEFT JOIN {agencies_servicelines_branches} asb ON asb.asb_slid = ce.epi_slid AND asb.asb_branchcode = tp.d_branchcode
LEFT JOIN {agencies} ag ON ag.agency_id = asb.asb_agencyid
LEFT JOIN {companies} c ON c.company_id = ag.agency_companyid
LEFT JOIN hchb_pps_invoices ppsi ON ppsi.invnum = hp.uc_invoiceid
WHERE r.rf_hasrefundinpostdates = 1;
"""
)

spark.sql(
    f"""
CREATE OR REPLACE TEMP VIEW hchb_results_combined AS
SELECT * FROM hchb_results_payments
UNION ALL
SELECT * FROM hchb_results_refunds;
"""
)


params = spark.sql("""
SELECT
  parmDateType,
  parmDepositOption,
  parmDateFrom,
  parmDateTo
FROM hchb_parameters
""").collect()[0]

parmDateType = params["parmDateType"]
parmDepositOption = params["parmDepositOption"]
parmDateFrom = params["parmDateFrom"]
parmDateTo = params["parmDateTo"]


if parmDepositOption == 1:
    spark.sql(f"""
    INSERT INTO {hchb_cash_invoice}(
    company,
    agency,
    branch,
    rap_payment,
    final_payment,
    other_payment,
    refund_payment,
    unapplied_cash,
    load_timestamp
    )
    SELECT
        company,
        agency,
        branch,
        SUM(COALESCE(rap_payment, 0))    AS rap_payment,
        SUM(COALESCE(final_payment, 0))  AS final_payment,
        SUM(COALESCE(other_payment, 0))  AS other_payment,
        SUM(COALESCE(refund_payment, 0)) AS refund_payment,
        SUM(COALESCE(unapplied_cash, 0)) AS unapplied_cash,
        current_timestamp()              AS load_timestamp
    FROM hchb_results_combined
    GROUP BY sortorder, company, agency, branch
    ORDER BY sortorder, company, agency, branch
    """)
else:
  spark.sql(
      f"""
  INSERT INTO {hchb_cash_invoice}
  SELECT
    company,
    agency,
    branch,
    client_name,
    medical_record_number,
    payor_type_desc,
    payor_source,
    invoice_number,
    rap_payment,
    final_payment,
    other_payment,
    refund_payment,
    unapplied_cash,
    deposit_date,
    deposit_number,
    check_number,
    DATE_FORMAT(date_entered, 'MM/dd/yyyy') AS date_entered,
    month_end_close_reporting_period,
    psid,
    current_timestamp() AS load_timestamp
  FROM hchb_results_combined
  ORDER BY 
    date_entered,
    sortorder,
    company,
    agency,
    branch,
    deposit_number,
    check_number,
    client_name;

  """
  )

In [0]:
spark.sql(
    f"""
CREATE OR REPLACE TEMP VIEW hchb_cash_invoice_base AS
SELECT 
  'HCHB' AS source_system,
  c.company,
  c.agency,
  a.agency_id,
  c.Client_Name as client_name,
  CASE 
    WHEN c.branch REGEXP '[A-Z]' THEN om.targetofficenumber
    ELSE c.branch 
  END AS target_office_number,
  c.Medical_Record_Number as medical_record_number,
  c.Payor_Types AS payor_types,
  c.Invoice__ as invoice_number,
  y.cltid,
  c.rap_payment as rap_payment,
  c.final_payment as final_payment,
  c.other_payment as other_payment,
  c.refund_payment as refund_payment,
  c.unapplied_cash as unapplied_cash,
  c.rap_payment + c.final_payment + c.other_payment - c.refund_payment + c.unapplied_cash AS total_cash,
  CASE 
    WHEN DAYOFWEEK(TO_DATE(c.date_entered, 'MM/dd/yyyy')) = 1 THEN TO_DATE(c.date_entered, 'MM/dd/yyyy')
    WHEN DAYOFWEEK(TO_DATE(c.date_entered, 'MM/dd/yyyy')) = 2 THEN DATE_ADD(TO_DATE(c.date_entered, 'MM/dd/yyyy'), -1)
    WHEN DAYOFWEEK(TO_DATE(c.date_entered, 'MM/dd/yyyy')) = 3 THEN DATE_ADD(TO_DATE(c.date_entered, 'MM/dd/yyyy'), -2)
    WHEN DAYOFWEEK(TO_DATE(c.date_entered, 'MM/dd/yyyy')) = 4 THEN DATE_ADD(TO_DATE(c.date_entered, 'MM/dd/yyyy'), -3)
    WHEN DAYOFWEEK(TO_DATE(c.date_entered, 'MM/dd/yyyy')) = 5 THEN DATE_ADD(TO_DATE(c.date_entered, 'MM/dd/yyyy'), 3)
    WHEN DAYOFWEEK(TO_DATE(c.date_entered, 'MM/dd/yyyy')) = 6 THEN DATE_ADD(TO_DATE(c.date_entered, 'MM/dd/yyyy'), 2)
    WHEN DAYOFWEEK(TO_DATE(c.date_entered, 'MM/dd/yyyy')) = 7 THEN DATE_ADD(TO_DATE(c.date_entered, 'MM/dd/yyyy'), 1)
  END AS reporting_week_ending_date,
  TO_DATE(c.deposit_date, 'MM/dd/yyyy') AS deposit_date,
  dd.datekey AS hchb_deposit_date_key,
  c.Deposit__ as deposit_number,
  c.Check__ as check_number,
  TO_DATE(c.date_entered, 'MM/dd/yyyy') AS date_entered,
  d.datekey AS hchb_date_entered_key,
  c.Month_End_Close_Reporting_Period as month_end_close_reporting_period,
  c.psid
FROM {stg_hchb_cash} c
LEFT JOIN {agencies} a 
  ON a.agency_name = c.agency
LEFT JOIN {officemapping} om 
  ON om.sourceofficecode = c.branch
LEFT JOIN {yrinvo} y 
  ON y.invnum = c.Invoice__
LEFT JOIN {datedimension} d 
  ON d.calendardate = TO_DATE(c.date_entered, 'MM/dd/yyyy')
LEFT JOIN {datedimension} dd 
  ON dd.calendardate = TO_DATE(c.deposit_date, 'MM/dd/yyyy');
"""
)

spark.sql(
    f"""
CREATE OR REPLACE TEMP VIEW hchb_cash_invoice_with_episodes AS
SELECT 
  cb.*,
  hcea.epi_paid,
  hcea.epi_id
FROM hchb_cash_invoice_base cb
LEFT JOIN (
  SELECT 
    epi_id,
    epi_paid,
    ROW_NUMBER() OVER (PARTITION BY epi_paid ORDER BY epi_paid) AS rnk
  FROM {client_episodes_all}
) hcea 
  ON CAST(cb.cltid AS STRING) = CAST(hcea.epi_paid AS STRING)
  AND hcea.rnk = 1;
"""
)

spark.sql(
    f"""
CREATE OR REPLACE TEMP VIEW hchb_cash_invoice_with_dimensions AS
SELECT 
  ss.sourcesystemkey,
  dad.datekey AS reporting_week_ending_date_key,
  cwe.agency_id,
  cwe.epi_paid,
  cwe.epi_id AS epid,
  o.officekey,
  cwe.medical_record_number,
  pd.payerkey,
  cwe.payor_types AS payor_category,
  cwe.invoice_number,
  cwe.rap_payment,
  cwe.final_payment,
  cwe.other_payment,
  cwe.refund_payment,
  cwe.unapplied_cash,
  cwe.total_cash,
  cwe.hchb_deposit_date_key,
  cwe.deposit_number,
  cwe.check_number,
  cwe.hchb_date_entered_key,
  cwe.month_end_close_reporting_period
FROM hchb_cash_invoice_with_episodes cwe
LEFT JOIN {office} o 
  ON o.officenumber = cwe.target_office_number
LEFT JOIN {datedimension} dad 
  ON dad.calendardate = cwe.reporting_week_ending_date
LEFT JOIN {cubeserviceofficetxnsourcesystem} ss 
  ON ss.sourcesystemname = cwe.source_system
LEFT JOIN {payerdimension} pd 
  ON pd.payerid = CAST(cwe.psid AS STRING);
"""
)

spark.sql(
    f"""
CREATE OR REPLACE TEMP VIEW cash_invoice_final AS
SELECT 
  cwd.sourcesystemkey,
  cwd.reporting_week_ending_date_key,
  cwd.agency_id,
  cwd.epi_paid,
  cwd.epid,
  csi.clientkey,
  cwd.officekey,
  cwd.medical_record_number,
  cwd.payerkey,
  cwd.payor_category,
  cwd.invoice_number,
  cwd.rap_payment,
  cwd.final_payment,
  cwd.other_payment,
  cwd.refund_payment,
  cwd.unapplied_cash,
  cwd.total_cash,
  cwd.hchb_deposit_date_key,
  cwd.deposit_number,
  cwd.check_number,
  cwd.hchb_date_entered_key,
  cwd.month_end_close_reporting_period
FROM hchb_cash_invoice_with_dimensions cwd
LEFT JOIN (
  SELECT * FROM (
    SELECT 
      sourcesystemid,
      clientkey,
      sourcesystem,
      ROW_NUMBER() OVER (
        PARTITION BY sourcesystemid, sourcesystem 
        ORDER BY sourcesystemid
      ) AS rnk1
    FROM {client}
  ) rc1
  WHERE rc1.rnk1 = 1
) csi 
  ON CAST(cwd.epid AS STRING) = LTRIM(RTRIM(CAST(csi.sourcesystemid AS STRING)))
  AND csi.sourcesystem = 'HCHB';
"""
)

In [0]:
spark.sql(
    f"""
    delete from {cash_invoice} where source_system_key=6
    """
)

In [0]:
spark.sql(
    f"""
INSERT INTO {cash_invoice}
(
    source_system_key,
    reporting_week_ending_date_key,
    posted_date_key,
    payor_key,
    client_key,
    cash_collected,
    deposit_date_key,
    batch_id,
    check_id,
    office_key,
    type,
    bank,
    agency_id,
    payor_category,
    invoice_number,
    rap_payment,
    final_payment,
    other_payment,
    refund_payment,
    unapplied_cash,
    deposit_number,
    hchb_date_entered_key,
    month_end_close_reporting_period
)
SELECT 
  sourcesystemkey AS source_system_key,
  reporting_week_ending_date_key,
  hchb_date_entered_key AS posted_date_key,
  payerkey AS payor_key,
  clientkey AS client_key,
  CAST(total_cash AS FLOAT) AS cash_collected,
  hchb_deposit_date_key AS deposit_date_key,
  NULL AS batch_id,
  check_number AS check_id,
  officekey AS office_key,
  NULL AS type,
  NULL AS bank,
  agency_id,
  payor_category,
  invoice_number,
  rap_payment,
  final_payment,
  other_payment,
  refund_payment,
  unapplied_cash,
  deposit_number,
  hchb_date_entered_key,
  month_end_close_reporting_period
FROM cash_invoice_final;
"""
)